# Analisis Sebaran Data TUSZ Segmentation

Notebook ini dibuat untuk melihat sebaran data hasil segmentasi dataset TUSZ. Kita akan memvisualisasikan jumlah sampel per kelas dan sebaran ukuran/durasi file.

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Konfigurasi tampilan plot
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Definisi path data
# Sesuaikan path ini dengan lokasi dataset yang sudah diproses
DATA_DIR = '/home/deepbrain/disertasi/code/tusz-segmentation/processed_data/train'

data = []

# Mengambil daftar folder (tipe kejang)
if os.path.exists(DATA_DIR):
    seizure_types = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"Tipe kejang yang ditemukan: {seizure_types}")

    for sz_type in seizure_types:
        folder_path = os.path.join(DATA_DIR, sz_type)
        # Mencari semua file .npy
        files = list(Path(folder_path).glob('*.npy'))
        
        for f in files:
            # Mengambil ukuran file sebagai proxy untuk durasi
            size_bytes = f.stat().st_size
            
            # Parsing nama file untuk info tambahan (opsional, tergantung format nama file)
            # Contoh: aaaaaacz_s006_t000_seg1.npy -> patient_id = aaaaaacz
            patient_id = f.name.split('_')[0]
            
            data.append({
                'seizure_type': sz_type,
                'filename': f.name,
                'patient_id': patient_id,
                'size_bytes': size_bytes
            })
else:
    print(f"Directory {DATA_DIR} tidak ditemukan!")

# Membuat DataFrame
df = pd.DataFrame(data)
df['size_kb'] = df['size_bytes'] / 1024
df['size_mb'] = df['size_kb'] / 1024

print(f"Total segmen data: {len(df)}")
df.head()

In [ ]:
# 1. Bar Chart: Jumlah Sampel per Tipe Kejang
if not df.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(x='seizure_type', data=df, palette='viridis', order=df['seizure_type'].value_counts().index)
    plt.title('Jumlah Sampel Segmen per Jenis Kejang')
    plt.xlabel('Jenis Kejang')
    plt.ylabel('Jumlah File')
    plt.xticks(rotation=45)
    
    # Menambahkan label angka di atas bar
    for i in ax.containers:
        ax.bar_label(i,)
        
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada data untuk ditampilkan.")

In [ ]:
# 2. Scatter Plot (Strip Plot): Sebaran Ukuran File per Tipe Kejang
# Ukuran file berbanding lurus dengan durasi/jumlah time steps
if not df.empty:
    plt.figure(figsize=(12, 8))
    sns.stripplot(x='seizure_type', y='size_mb', data=df, alpha=0.6, jitter=0.2, palette='coolwarm', order=df['seizure_type'].value_counts().index)
    plt.title('Sebaran Ukuran File (Indikasi Durasi) per Jenis Kejang')
    plt.xlabel('Jenis Kejang')
    plt.ylabel('Ukuran File (MB)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada data untuk ditampilkan.")

In [ ]:
# 3. Box Plot: Statistik Ukuran File per Tipe Kejang
if not df.empty:
    plt.figure(figsize=(12, 8))
    sns.boxplot(x='seizure_type', y='size_mb', data=df, palette='Set2', order=df['seizure_type'].value_counts().index)
    plt.title('Distribusi Statistik Ukuran File per Jenis Kejang')
    plt.xlabel('Jenis Kejang')
    plt.ylabel('Ukuran File (MB)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada data untuk ditampilkan.")